# JuaKazi `ki-bias-corrector-v1` — Training Notebook

**Model:** `juakazike/ki-bias-corrector-v1`
**Base:** `google/mt5-small` (mT5 — best available for Kikuyu)
**Task:** Seq2seq gender bias correction
**Runtime:** GPU T4 · ~2 hours
**Target:** BLEU >= 15 (lower than HA/ZU — Kikuyu mT5 has weaker pretraining)

### Data source
`eval/ground_truth_ki_v8.csv` — 11,622 rows, all have `expected_correction`.
Filter to has_bias=True rows only (1,603 pairs).

### Before running
1. Accelerator → GPU
2. Upload `ground_truth_ki_v8.csv` to Kaggle dataset `juakazi-ki-corrector`
3. Set `HF_TOKEN` in Kaggle Secrets


In [ ]:
# Cell 1: Install
import subprocess, sys
r = subprocess.run([sys.executable,'-m','pip','install','--upgrade',
    'transformers>=4.38.0','sentencepiece','sacrebleu','accelerate>=0.27.0',
    'huggingface_hub>=0.20.0','numpy<2.0.0'], capture_output=True, text=True)
print('OK' if r.returncode==0 else r.stderr)
print('Restart then run Cell 2.')

In [ ]:
# Cell 2: Verify
import torch, transformers
print(f'torch: {torch.__version__}  GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"}')
assert torch.cuda.is_available()

In [ ]:
# Cell 3: Load KI correction pairs from ground truth
import csv, os

GT_CSV = '/kaggle/input/juakazi-ki-corrector/ground_truth_ki_v8.csv'
assert os.path.exists(GT_CSV), f'Not found: {GT_CSV}'

pairs = []
with open(GT_CSV, encoding='utf-8') as f:
    for r in csv.DictReader(f):
        hb  = str(r.get('has_bias','')).strip().lower()
        src = r.get('text','').strip()
        tgt = r.get('expected_correction','').strip()
        if hb in ('true','1') and src and tgt and src != tgt:
            pairs.append((src, tgt))

print(f'Loaded {len(pairs)} KI correction pairs from ground truth')
print(f'Sample: {pairs[0]}')

In [ ]:
# Cell 4: Config
import random, numpy as np
from pathlib import Path

SEED       = 42
BASE_MODEL = 'google/mt5-small'  # best available for Kikuyu
OUTPUT_DIR = '/kaggle/working/output'
MAX_SRC    = 128
MAX_TGT    = 128
EPOCHS     = 10
BATCH      = 8
GRAD_ACCUM = 4
LR         = 3e-4
REPO_ID    = 'juakazike/ki-bias-corrector-v1'
PREFIX     = 'correct bias: '

random.seed(SEED); np.random.seed(SEED)
import torch; torch.manual_seed(SEED)
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

random.shuffle(pairs)
n = len(pairs)
train_pairs = pairs[:int(n*0.80)]
val_pairs   = pairs[int(n*0.80):int(n*0.90)]
test_pairs  = pairs[int(n*0.90):]

print(f'BASE: {BASE_MODEL}')
print(f'Train: {len(train_pairs)}  Val: {len(val_pairs)}  Test: {len(test_pairs)}')

In [ ]:
# Cell 5: Tokenizer + Dataset
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

class CorrectionDataset(Dataset):
    def __init__(self, pairs, tokenizer, max_src, max_tgt):
        self.pairs=pairs; self.tok=tokenizer; self.ms=max_src; self.mt=max_tgt
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        src, tgt = self.pairs[idx]
        inp = self.tok(PREFIX+src, max_length=self.ms, padding='max_length',
                       truncation=True, return_tensors='pt')
        lbl = self.tok(tgt, max_length=self.mt, padding='max_length',
                       truncation=True, return_tensors='pt').input_ids
        lbl[lbl == self.tok.pad_token_id] = -100
        return {'input_ids':inp.input_ids.squeeze(0),
                'attention_mask':inp.attention_mask.squeeze(0),
                'labels':lbl.squeeze(0)}

train_ds=CorrectionDataset(train_pairs,tokenizer,MAX_SRC,MAX_TGT)
val_ds  =CorrectionDataset(val_pairs,  tokenizer,MAX_SRC,MAX_TGT)
test_ds =CorrectionDataset(test_pairs, tokenizer,MAX_SRC,MAX_TGT)
print(f'Ready. Train={len(train_ds)} Val={len(val_ds)} Test={len(test_ds)}')

In [ ]:
# Cell 6: Model + Trainer
import torch, numpy as np
from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments
from transformers import DataCollatorForSeq2Seq, EarlyStoppingCallback

model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)
print(f'mT5-small loaded: {sum(p.numel() for p in model.parameters())/1e6:.0f}M params')
collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

def compute_bleu(eval_pred):
    import sacrebleu
    preds, labels = eval_pred
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    dp = tokenizer.batch_decode(preds,  skip_special_tokens=True)
    dl = tokenizer.batch_decode(labels, skip_special_tokens=True)
    score = sacrebleu.corpus_bleu(dp, [dl]).score
    print(f'  BLEU={score:.2f}')
    return {'bleu': round(score, 2)}

args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR, num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH, per_device_eval_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACCUM, learning_rate=LR,
    warmup_ratio=0.10, weight_decay=0.01,
    eval_strategy='epoch', save_strategy='epoch',
    load_best_model_at_end=True, metric_for_best_model='bleu',
    greater_is_better=True, predict_with_generate=True,
    fp16=torch.cuda.is_available(), logging_steps=50, seed=SEED, report_to='none',
)
trainer = Seq2SeqTrainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_ds,
    tokenizer=tokenizer, data_collator=collator,
    compute_metrics=compute_bleu,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)
print('Trainer ready.')

In [ ]:
# Cell 7: TRAIN
result = trainer.train()
print(f'Done. Steps={result.global_step}  Loss={result.training_loss:.4f}')

In [ ]:
# Cell 8: Evaluate + smoke test
import torch, sacrebleu, numpy as np
pred_out = trainer.predict(test_ds)
labels   = np.where(pred_out.label_ids!=-100, pred_out.label_ids, tokenizer.pad_token_id)
dp = tokenizer.batch_decode(pred_out.predictions, skip_special_tokens=True)
dl = tokenizer.batch_decode(labels, skip_special_tokens=True)
bleu = sacrebleu.corpus_bleu(dp, [dl]).score
target_met = bleu >= 15.0
print(f'Test BLEU: {bleu:.2f}  (target >=15)  {"✓" if target_met else "✗"}')

print('\n--- Sample KI corrections ---')
for i in range(min(5, len(test_pairs))):
    src, ref = test_pairs[i]
    gen = trainer.model.generate(
        tokenizer(PREFIX+src, return_tensors='pt', max_length=MAX_SRC, truncation=True).input_ids.cuda())
    out = tokenizer.decode(gen[0], skip_special_tokens=True)
    print(f'  IN:  {src[:80]}\n  OUT: {out[:80]}\n  REF: {ref[:80]}\n')

In [ ]:
# Cell 9: Save + upload
import json, os
trainer.save_model(OUTPUT_DIR); tokenizer.save_pretrained(OUTPUT_DIR)
meta = {'model_id':REPO_ID,'base_model':BASE_MODEL,'language':'ki',
        'test_bleu':round(bleu,2),'target_met':target_met,
        'train_pairs':len(train_pairs),'prefix':PREFIX}
with open(f'{OUTPUT_DIR}/training_meta.json','w') as f: json.dump(meta,f,indent=2)
print(json.dumps(meta,indent=2))

HF_TOKEN = os.environ.get('HF_TOKEN')
if HF_TOKEN and target_met:
    from huggingface_hub import HfApi
    api = HfApi()
    api.create_repo(REPO_ID, token=HF_TOKEN, exist_ok=True, private=False)
    api.upload_folder(folder_path=OUTPUT_DIR, repo_id=REPO_ID, token=HF_TOKEN)
    print(f'Uploaded → https://huggingface.co/{REPO_ID}')
elif not target_met:
    print('BLEU below target. Increase epochs or check data quality.')
else:
    print('Set HF_TOKEN in Kaggle Secrets.')